In [ ]:
%load_ext autoreload
%autoreload 2

import torch

from face_verif_gen.model import Block, IResNet

In [ ]:
model = IResNet()
ckpt = torch.load("../checkpoints/adaface_ir18_casia.ckpt", map_location="cpu", weights_only=True)
# ckpt.keys()['state_dict']
ckpt["state_dict"].keys()

In [ ]:
b1 = Block(64, 64, stride=2)
img = torch.randn(1, 64, 112, 112)
b1(img).shape

In [ ]:
i1 = IResNet()
i1(torch.randn(1, 3, 112, 112))
for param in i1.parameters():
    print(param.numel())

In [ ]:
def count_params(module):
    return sum(param.numel() for param in module.parameters())


def print_breakdown(module, name="IResNet", indent=0):
    print(f"{'  ' * indent}{name} ({type(module).__name__}): {count_params(module):,}")
    for child_name, child in module.named_children():
        print_breakdown(child, child_name, indent + 1)


print_breakdown(i1)
print(f"\nTotal parameters: {count_params(i1):,}")

In [ ]:
for name, param in i1.body.named_parameters():
    print(name, param.numel())

In [ ]:
for name, param in i1.output.named_parameters():
    print(name, param.numel())

In [ ]:
b2 = Block(64, 128, stride=2)
b2(img).shape

In [ ]:
from face_verif_gen.heads import SoftmaxHead

torch.manual_seed(42)

NUM_CLASSES = 9072  # people in the training split: one logit, and one weight row, each
embeddings = torch.randn(8, 512)  # stand-in for IResNet's output: 8 images, 512 features each
labels = torch.randint(0, NUM_CLASSES, (8,))  # each image's true identity, 0 to 9071
head = SoftmaxHead(512, NUM_CLASSES)  # nn.Linear(512, 9072, bias=False)
logits = head(embeddings, labels)  # (8, 9072): each image's score for every identity

embeddings.shape, head.fc.weight.shape, logits.shape

In [ ]:
def cross_entropy_loss(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """Compute the cross-entropy loss from logits and labels, in fp32."""
    logits = logits.float()  # exp and a sum over 9,072 classes lose too much in bf16
    # e^z overflows fp32 once z passes ~88. Subtracting each row's max changes nothing (it
    # cancels inside log softmax) but makes the largest term e^0 = 1 and the sum at least 1.
    row_max = logits.max(dim=1, keepdim=True).values
    shifted_logits = logits - row_max
    log_exponent_sums = torch.log(torch.exp(shifted_logits).sum(dim=1, keepdim=True))
    log_probs = shifted_logits - log_exponent_sums  # (B, C): log softmax of every class
    # Each row keeps only its true class: row i, column labels[i].
    true_log_probs = log_probs.gather(1, labels[:, None]).squeeze(1)  # (B,)
    return true_log_probs, log_probs[0, labels[0]]


cross_entropy_loss(logits, labels)

In [ ]:
labels, labels[:, None]